# `POST /validate-config` — assignment examples

Manual checks against the running server. Each cell sends one of the three input/output examples from `specs/assignment.md` to `POST /validate-config` and prints the prettified JSON response.

**Prereqs**

- Server running locally: `npm run dev:server` (or `npm run dev`).
- `OPENAI_API_KEY` set in `server/.env` (otherwise the endpoint returns `502`).
- Python `requests` available: `pip install requests`.

In [1]:
import json
import requests

BASE_URL = "http://localhost:3000"
ENDPOINT = f"{BASE_URL}/validate-config"
MODEL = "gpt-5"  # e.g. "gpt-4o-mini" or "gpt-4o"; None uses the server default

def call_validate(config: dict, model: str | None = MODEL) -> None:
    """POST `config` to /validate-config and pretty-print the response."""
    params = {"model": model} if model else None
    print("Request:")
    print(json.dumps(config, indent=2))
    print()
    response = requests.post(ENDPOINT, json=config, params=params, timeout=120)
    print(f"HTTP {response.status_code}")
    try:
        body = response.json()
        print(json.dumps(body, indent=2, ensure_ascii=False))
    except ValueError:
        print(response.text)

### The results are according to the following reference ranges
```json
{
  "difficulties": {
    "easy": {
      "reward_min": 100,
      "reward_max": 500,
      "time_limit_min": 30
    },
    "medium": {
      "reward_min": 500,
      "reward_max": 2000,
      "time_limit_min": 20,
      "time_limit_max": 60
    },
    "hard": {
      "reward_min": 2000,
      "reward_max": 5000,
      "time_limit_min": 10,
      "time_limit_max": 30
    }
  },
  "total_levels": 150
}

```

## Example 1 — reward too high for an easy level

Expected pattern: schema is valid; the LLM should flag a `reward_vs_difficulty` mismatch (5000 reward on `easy`).

In [2]:
call_validate({
    "level": 12,
    "time_limit": 60,
    "reward": 5000,
    "difficulty": "easy"
})

Request:
{
  "level": 12,
  "time_limit": 60,
  "reward": 5000,
  "difficulty": "easy"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Early-game level 12 is marked easy with a generous 60s timer, which fits easy timing. However, the reward is set to 5000, which is the hard-tier maximum and far above the easy band, creating a strong mismatch between declared difficulty and payout.",
    "suggested_actions": [
      "Reduce reward to 100–500 for easy difficulty."
    ],
    "confidence": 0.96
  }
}


## Example 2 — time limit too tight for a hard level

Expected pattern: schema is valid; the LLM should flag `time_vs_difficulty` (10s on `hard`) and likely `frustration_risk`.

In [3]:
call_validate({
    "level": 5,
    "time_limit": 10,
    "reward": 500,
    "difficulty": "hard"
})

Request:
{
  "level": 5,
  "time_limit": 10,
  "reward": 500,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Very early level (5/150) is labeled hard, with a minimal hard-tier timer of 10s and a low reward of 500. The timer fits the hard band, but the reward is in the easy/medium area, and the early placement contradicts a hard difficulty spike.",
    "suggested_actions": [
      "Increase reward to 2000-5000 for hard difficulty, or lower the difficulty to match a 500 reward.",
      "Reclassify this stage to easy (or move it much later if it must remain hard)."
    ],
    "confidence": 0.92
  }
}


## Example 3 — reasonable starting level (expect empty findings)

Expected pattern: schema is valid; the LLM should return an empty `findings` array, so `suggested_actions` is `["No action needed"]` and `confidence` is the model's `verdict_confidence`.

In [4]:
call_validate({
    "level": 1,
    "time_limit": 120,
    "reward": 100,
    "difficulty": "easy"
})

Request:
{
  "level": 1,
  "time_limit": 120,
  "reward": 100,
  "difficulty": "easy"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 1 is marked easy with a very generous 120s timer and a reward at the low end of the easy band (100). Both reward and time fall within the Easy ranges (time has only a minimum bound), and the earliest progression position aligns with easy difficulty. No economy distortion or coherence conflicts are apparent.",
    "suggested_actions": [
      "No action needed"
    ],
    "confidence": 0.95
  }
}


## Bonus — schema-validation failure (expect HTTP 400, no LLM call)

Sends a malformed body to confirm the Zod gate rejects it before the LLM is called.

In [5]:
call_validate({"level": "oops"})

Request:
{
  "level": "oops"
}

HTTP 400
{
  "schema_validation": {
    "valid": false,
    "errors": [
      {
        "path": "level",
        "message": "Expected number, received string"
      },
      {
        "path": "time_limit",
        "message": "Required"
      },
      {
        "path": "reward",
        "message": "Required"
      },
      {
        "path": "difficulty",
        "message": "Required"
      }
    ]
  }
}


In [6]:
call_validate({
    "level": 150,
    "time_limit": 25,
    "reward": 4500,
    "difficulty": "hard"
})

Request:
{
  "level": 150,
  "time_limit": 25,
  "reward": 4500,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Final level set to hard difficulty with a 25s timer and 4500 reward. Both reward and time fall squarely within the hard ranges, and placing hard at the very end aligns with expected progression. Nothing suggests economy distortion or a frustrating mix for a capstone level.",
    "suggested_actions": [
      "No action needed"
    ],
    "confidence": 0.96
  }
}
